# Multiple Waveform Synchronous Playback
Plays a square pulse on QRM sequencer 0, QCM sequencer 0, and QCM sequencer 1. Qblox SYNQ functionality is used to precisely play pulses, such that pulse 2 begins halfway through pulse 1, and pulse 3 begins halfway through pulse 2. The outputs of Qblox are viewed on an oscilloscope with a 1MOhm input impedance. Pulses are played on O1 of the QRM, and O1 and O3 of the QCM.

Authors: Noah Stieler, Luke Dyer

In [1]:
import json

from qblox_instruments import Cluster
from qblox_instruments.types import InstrumentType

In [2]:
#Scan for clusters
!qblox-pnp list

Devices:
 - 192.168.137.2: cluster_mm 0.6.2 with name "cluster-mm" and serial number 00015_2251_003


#### Configure and connect to Qblox modules

In [3]:
cluster = Cluster("cluster_mm", "192.168.137.2")
moduleQRM = None
moduleQCM = None

modules = [mod for mod in cluster.modules if mod.present()]
for mod in modules:
	print(str(mod) + " type=" + str(mod.module_type) + " is_rf_type=" + str(mod.is_rf_type))

cluster.reset()
cluster.get_system_state()

moduleQRM = modules[1]
moduleQCM = modules[0]

moduleQRM.disconnect_outputs()
moduleQRM.disconnect_inputs()
moduleQRM.sequencer0.connect_sequencer("out0_1")	
moduleQRM.sequencer0.sync_en(True)						#Enable wait_sync Q1ASM instruction

moduleQCM.disconnect_outputs()
moduleQCM.sequencer0.connect_sequencer("out0_1")
moduleQCM.sequencer0.sync_en(True)						#Enable wait_sync Q1ASM instruction
moduleQCM.sequencer1.connect_sequencer("out2_3")
moduleQCM.sequencer1.sync_en(True)						#Enable wait_sync Q1ASM instruction

<QcmQrm: cluster_mm_module2 of Cluster: cluster_mm> type=QCM is_rf_type=False
<QcmQrm: cluster_mm_module4 of Cluster: cluster_mm> type=QRM is_rf_type=False
<QcmQrm: cluster_mm_module6 of Cluster: cluster_mm> type=QCM is_rf_type=True


#### Define Q1ASM sequences

In [4]:
#Length are in nanoseconds
waveform1Length = 4000
waveform2Length = 8000
waveform3Length = 12000

sequence1 = f"""
	wait_sync	4
	play		0,0,16384
	stop
"""
#Start playing waveform2 halfway through waveform1
sequence2 = f"""
	wait_sync	4
	wait		{(int)(waveform1Length/2.0)}
	play		0,0,16384
	stop
"""
#Start playing waveform3 halfway through waveform2
sequence3 = f"""
	wait_sync	4
	wait		{(int)((waveform1Length/2.0) + (waveform2Length/2.0))}
	play		0,0,16384
	stop
"""

#### Generate JSON files and upload them to sequencers

In [5]:
def generateSequence(waveform:list[float], sequenceProgram:str) -> dict:
	#Qblox requires this formatting for each of these dictionaries
	waveforms = {
		"block": {"data": waveform, "index": 0,}
	}
	acquisitions = {
		"bin": {"num_bins": 1, "index": 0}	
	}
	sequence = {
		"waveforms": waveforms,
		"weights": {},
		"acquisitions": acquisitions,
		"program": sequenceProgram,
	}
	return sequence

#Create JSON files containing all the information that gets uploaded to each sequencer
s1 = generateSequence([1.0 for i in range(waveform1Length)], sequence1)
#For a 50Ohm load, the QCM module has a maximum output of -2.5V and waveform amplitudes are 
#sent as percentages of maxmimum value. Since we are using a 1MOhm load, 0.2 amplitude
#Gives a 1V output into the load.
s2 = generateSequence([0.2 for i in range(waveform2Length)], sequence2)
s3 = generateSequence([0.2 for i in range(waveform3Length)], sequence3)

#Upload the JSON files
moduleQRM.sequencer0.sequence(s1)
moduleQCM.sequencer0.sequence(s2)
moduleQCM.sequencer1.sequence(s3)

#### Start sequencers

In [6]:
moduleQRM.arm_sequencer(0)
moduleQCM.arm_sequencer(0)
moduleQCM.arm_sequencer(1)

moduleQRM.start_sequencer()
moduleQCM.start_sequencer()

print(moduleQRM.get_sequencer_state(0))
print(moduleQCM.get_sequencer_state(0))
print(moduleQCM.get_sequencer_state(1))

Status: STOPPED, Flags: NONE
Status: STOPPED, Flags: NONE
Status: STOPPED, Flags: NONE
